In [2]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
import os
import kagglehub
import pandas as pd

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
def extract_mfccs(data,sample_rate):
    mfccs = librosa.feature.mfcc(y=data, sr=sample_rate, n_mfcc=15).T     # Extract 15 MFCCs of the form of a 2D array
    return mfccs

def extract_chroma(data,sample_rate):
    chroma = librosa.feature.chroma_stft(y=data, sr=sample_rate, n_chroma=12).T
    return chroma

def extract_mel(data,sample_rate):
    mel = librosa.feature.melspectrogram(y=data, sr=sample_rate, n_mels=128).T
    return mel

In [28]:
def noiseInjection(data):
    signal_power = np.mean(data**2)
    noise_power = signal_power * 0.0032     # calculated for an approximate SNR of 25 dB
    noise = np.random.normal(0,np.sqrt(noise_power),len(data))
    augmented_data = data + noise
    return augmented_data

def shifting(data):
    shift = int(np.random.uniform(-0.1,0.1)*len(data))
    augmented_data = np.roll(data,shift)
    return augmented_data

# no pitch stretching since this is fine-tuning to specific people


In [29]:
current_dir = os.getcwd()
audio = os.path.join(current_dir, "audio")
dirlist = os.listdir(audio)
print(dirlist)

['.DS_Store', 'female-sad', 'female-angry', 'male-angry', 'male-disgust', 'female-happy', 'female-disgust', 'female-neutral', 'female-fearful', 'male-happy', 'male-sad', 'female-surprised', 'male-surprised', 'male-neutral', 'male-fearful']


In [30]:
emotions = []
genders = []
labels = []
mfccs_list = []
chroma_list = []
mel_list = []


for subfolder in dirlist:
    subfolder_path = os.path.join(audio, subfolder)
    print(f"Processing subfolder: {subfolder}")
    if os.path.isdir(subfolder_path):
        for file in os.listdir(subfolder_path):
            if file.endswith('.wav'):
                print(f"Processing file: {file}")
                file_path = os.path.join(subfolder_path, file)
                data1, sample_rate = librosa.load(file_path)
                data2 = noiseInjection(data1)
                data3 = shifting(data1)
                for data in [data1, data2, data3]:
                    mfccs = extract_mfccs(data, sample_rate)
                    chroma = extract_chroma(data, sample_rate)
                    mel = extract_mel(data, sample_rate)
                    mfccs_list.append(mfccs)
                    chroma_list.append(chroma)
                    mel_list.append(mel)
                    emotion = subfolder.split("-")[1]
                    emotions.append(emotion)
                    gender = subfolder.split("-")[0]
                    genders.append(gender)
                    labels.append(subfolder)
print("Emotions:", emotions)
print("Genders:", genders)
print("Labels:", labels)


Processing subfolder: .DS_Store
Processing subfolder: female-sad
Processing file: 5.wav
Processing file: 4.wav


/var/folders/pl/_6vmbhxs2r33l70nwj38v9pw0000gn/T/ipykernel_47080/3601344001.py:17: UserWarning: PySoundFile failed. Trying audioread instead.
  data1, sample_rate = librosa.load(file_path)


Processing file: 1.wav
Processing file: 3.wav
Processing file: 2.wav
Processing subfolder: female-angry
Processing file: 5.wav
Processing file: 4.wav
Processing file: 1.wav
Processing file: 3.wav
Processing file: 2.wav
Processing subfolder: male-angry
Processing file: 5.wav
Processing file: 4.wav
Processing file: 1.wav
Processing file: 3.wav
Processing file: 2.wav
Processing subfolder: male-disgust
Processing file: 5.wav
Processing file: 4.wav
Processing file: 1.wav
Processing file: 3.wav
Processing file: 2.wav
Processing subfolder: female-happy
Processing file: 5.wav
Processing file: 4.wav
Processing file: 1.wav
Processing file: 3.wav
Processing file: 2.wav
Processing subfolder: female-disgust
Processing file: 5.wav
Processing file: 4.wav
Processing file: 1.wav
Processing file: 3.wav
Processing file: 2.wav
Processing subfolder: female-neutral
Processing file: 5.wav
Processing file: 4.wav
Processing file: 1.wav
Processing file: 3.wav
Processing file: 2.wav
Processing subfolder: female-

In [31]:
df = pd.DataFrame([labels,mfccs_list,chroma_list,mel_list]).T
df.columns = ["Label", "MFCCs", "Chroma", "Mel"]
print(df.head())
print(df.tail())
print(df.shape)


        Label                                              MFCCs  \
0  female-sad  [[-692.21954, 79.350464, 16.638435, 10.515955,...   
1  female-sad  [[-563.0235518727306, 16.015328940259874, 10.1...   
2  female-sad  [[-680.89355, 92.61116, 14.225828, -13.32783, ...   
3  female-sad  [[-665.1343, 105.103806, 4.6220074, 20.59402, ...   
4  female-sad  [[-540.124160551421, 18.579072831727608, 4.851...   

                                              Chroma  \
0  [[0.775319, 0.607652, 0.6237509, 0.34317195, 0...   
1  [[1.0, 0.9293065170679017, 0.9502667384324928,...   
2  [[0.5057444, 0.367797, 0.3380883, 0.43130913, ...   
3  [[1.0, 0.6071119, 0.35067463, 0.12454112, 0.12...   
4  [[1.0, 0.6199532122470834, 0.42178236492961857...   

                                                 Mel  
0  [[4.0646537e-06, 3.3337237e-05, 7.479243e-05, ...  
1  [[5.483052901391637e-06, 1.8957323791490546e-0...  
2  [[5.4673396e-06, 4.4179476e-05, 2.9605466e-05,...  
3  [[9.494077e-06, 1.2984458e-05, 

In [33]:
numpy_array = df.to_numpy()
np.save('labels_fine_tuning.npy', numpy_array)